# 🔄 Step 2b: Silver & Gold Transformations (Bronze → Silver → Gold)

Runs after Auto Loader (`02a_autoloader_ingest`) as a task in the **Lakeflow Job**
`soc-triage-pipeline`:

- **Silver**: parse JSON payloads, OCSF-aligned fields, deduplicate on `event_id`
- **Gold**: enrich with threat intel, asset and customer context; composite risk score and tier

In [0]:
%run ./01_config

# ⚙️ Configuration — SOC Alert Triage

Shared configuration for all notebooks. Update the catalog name below before running.

✅ Configuration loaded
   Catalog: rajarora_febar
   Schema:  soc_alert_triage
   Tables:  10 tables configured


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
bronze_df = spark.table(TABLES["raw_security_events"])
print(f"📊 Bronze layer: {bronze_df.count():,} raw security events")
bronze_df.select("event_id", "event_timestamp", "source_type", "customer_id", "severity_score").display()

📊 Bronze layer: 500,000 raw security events


event_id,event_timestamp,source_type,customer_id,severity_score
04d17e69-2558-4779-9dcd-f2414ab352ed,2026-09-01T17:35:55Z,cloud_audit,CUST-0010,8
5dc6ee1b-1aa6-4693-8e99-2ea3eae02638,2026-09-01T02:39:41Z,cloud_audit,CUST-0041,4
2bc06bf7-b47f-4aa8-bcb9-a021ff0205db,2026-09-01T16:45:52Z,firewall,CUST-0058,8
704b2a1e-05c6-4cb1-bfd9-36eed6ac68d4,2026-09-01T06:20:46Z,auth,CUST-0081,2
b071e559-1915-468c-b260-c5aa7adc9ce5,2026-09-01T21:51:38Z,firewall,CUST-0089,3
2b10385e-c3aa-42dc-9140-f41923653f2d,2026-09-01T21:21:25Z,firewall,CUST-0047,2
5fe8c38c-5687-4228-bde6-be45747c517e,2026-09-01T04:37:08Z,firewall,CUST-0061,3
d44d7789-2c06-46a8-b782-ea61be13a535,2026-09-01T20:23:49Z,firewall,CUST-0029,2
ad5eee60-1674-47cd-977f-83ca5f5e558f,2026-09-01T23:11:46Z,cloud_audit,CUST-0077,1
0d82d595-8a1a-4e88-90c2-b6a7e5b9652a,2026-09-01T19:13:43Z,firewall,CUST-0055,8


In [0]:
silver_df = (
    bronze_df
    .withColumn("parsed_payload", F.from_json(F.col("raw_payload"), MapType(StringType(), StringType())))
    .withColumn("event_date", F.to_date(F.col("event_timestamp")))
    .withColumn("event_hour", F.hour(F.to_timestamp(F.col("event_timestamp"))))
    .withColumn("is_business_hours", 
                 (F.col("event_hour").between(8, 17)) & 
                 (F.dayofweek(F.col("event_date")).between(2, 6)))
    # Extract OCSF-aligned fields
    .withColumn("action", F.col("parsed_payload").getItem("action"))
    .withColumn("src_ip", F.coalesce(
        F.col("parsed_payload").getItem("src_ip"),
        F.col("parsed_payload").getItem("source_ip")
    ))
    .withColumn("dst_ip", F.col("parsed_payload").getItem("dst_ip"))
    .withColumn("username", F.coalesce(
        F.col("parsed_payload").getItem("username"),
        F.col("parsed_payload").getItem("principal")
    ))
    .withColumn("alert_type", F.col("parsed_payload").getItem("alert_type"))
    .withColumn("mitre_tactic", F.col("parsed_payload").getItem("mitre_tactic"))
    .withColumn("process_name", F.col("parsed_payload").getItem("process_name"))
    .withColumn("geo_location", F.col("parsed_payload").getItem("geo_location"))
    .withColumn("verdict", F.col("parsed_payload").getItem("verdict"))
    .withColumn("protocol", F.col("parsed_payload").getItem("protocol"))
    .withColumn("cloud_service", F.col("parsed_payload").getItem("service"))
    .withColumn("cloud_action", F.col("parsed_payload").getItem("action"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .drop("raw_payload", "parsed_payload")
    .dropDuplicates(["event_id"])
)

# Write Silver table
silver_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLES["normalized_events"])
silver_count = spark.table(TABLES["normalized_events"]).count()
print(f"✅ Silver layer: {silver_count:,} normalized events written")

# Show quality metrics
print("\n📊 Silver Layer Quality Metrics:")
spark.sql(f"""
    SELECT 
        source_type,
        COUNT(*) as events,
        COUNT(DISTINCT customer_id) as customers,
        ROUND(AVG(severity_score), 2) as avg_severity,
        SUM(CASE WHEN src_ip IS NOT NULL THEN 1 ELSE 0 END) as has_src_ip,
        SUM(CASE WHEN is_business_hours THEN 1 ELSE 0 END) as business_hours_events
    FROM {TABLES['normalized_events']}
    GROUP BY source_type
    ORDER BY events DESC
""").display()

✅ Silver layer: 500,000 normalized events written

📊 Silver Layer Quality Metrics:


source_type,events,customers,avg_severity,has_src_ip,business_hours_events
firewall,175021,100,5.65,175021,53430
auth,124891,100,4.7,124891,38138
edr,99905,100,6.59,0,30670
email,50145,100,5.17,0,15506
cloud_audit,50038,100,5.18,0,15291


In [0]:
# Load reference tables
threat_intel = spark.table(TABLES["threat_intel_iocs"]).filter("is_active = true")
assets = spark.table(TABLES["asset_inventory"])
customers = spark.table(TABLES["customers"])
silver = spark.table(TABLES["normalized_events"])

# Threat intel enrichment — match source IPs against known-bad IOCs
ip_iocs = threat_intel.filter("ioc_type = 'ip_address'").select(
    F.col("ioc_value").alias("matched_ioc"),
    F.col("threat_category"),
    F.col("confidence").alias("ioc_confidence")
)

gold_df = (
    silver
    # Join with threat intel on source IP
    .join(F.broadcast(ip_iocs), silver.src_ip == ip_iocs.matched_ioc, "left")
    # Join with asset inventory
    .join(
        assets.select("asset_id", "asset_type", "criticality", "department", "os_type"),
        "asset_id", "left"
    )
    # Join with customer info
    .join(
        customers.select("customer_id", "customer_name", "industry", "tier", "region"),
        "customer_id", "left"
    )
    # Compute enrichment flags
    .withColumn("threat_intel_match", F.when(F.col("matched_ioc").isNotNull(), True).otherwise(False))
    .withColumn("criticality_score", 
        F.when(F.col("criticality") == "critical", 4)
         .when(F.col("criticality") == "high", 3)
         .when(F.col("criticality") == "medium", 2)
         .otherwise(1))
    # Composite risk score (weighted formula)
    .withColumn("composite_risk_score", 
        F.round(
            (F.col("severity_score") * 0.4) +
            (F.when(F.col("threat_intel_match"), 10).otherwise(0) * 0.3) +
            (F.col("criticality_score") * 2.5 * 0.2) +
            (F.when(~F.col("is_business_hours"), 2).otherwise(0) * 0.1),
            2
        ))
    .withColumn("risk_tier",
        F.when(F.col("composite_risk_score") >= 7, "critical")
         .when(F.col("composite_risk_score") >= 5, "high")
         .when(F.col("composite_risk_score") >= 3, "medium")
         .otherwise("low"))
    .withColumn("enrichment_timestamp", F.current_timestamp())
    .drop("matched_ioc")
)

# Write Gold table. The RLS row filter must be dropped to overwrite the table, so it is re-applied
# immediately after the write (03_governance creates the filter function on the first run and re-asserts it).
RLS_FUNCTION = f"{FULL_SCHEMA}.rls_customer_filter"
if spark.catalog.tableExists(TABLES["enriched_alerts"]):
    spark.sql(f"ALTER TABLE {TABLES['enriched_alerts']} DROP ROW FILTER")
gold_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLES["enriched_alerts"])
if spark.sql(f"""SELECT 1 FROM {CATALOG_NAME}.information_schema.routines
               WHERE routine_schema = '{SCHEMA_NAME}' AND routine_name = 'rls_customer_filter'""").count() > 0:
    spark.sql(f"ALTER TABLE {TABLES['enriched_alerts']} SET ROW FILTER {RLS_FUNCTION} ON (customer_id)")
    print("🔒 Row filter re-applied to gold_enriched_alerts")
gold_count = spark.table(TABLES["enriched_alerts"]).count()
print(f"✅ Gold layer: {gold_count:,} enriched alerts written")

# Show enrichment summary
print("\n📊 Gold Layer Enrichment Summary:")
spark.sql(f"""
    SELECT 
        risk_tier,
        COUNT(*) as alert_count,
        ROUND(AVG(composite_risk_score), 2) as avg_risk_score,
        SUM(CASE WHEN threat_intel_match THEN 1 ELSE 0 END) as threat_intel_matches,
        COUNT(DISTINCT customer_id) as affected_customers
    FROM {TABLES['enriched_alerts']}
    GROUP BY risk_tier
    ORDER BY avg_risk_score DESC
""").display()

🔒 Row filter re-applied to gold_enriched_alerts
✅ Gold layer: 500,000 enriched alerts written

📊 Gold Layer Enrichment Summary:


risk_tier,alert_count,avg_risk_score,threat_intel_matches,affected_customers
critical,11688,7.98,11688,100
high,72268,5.47,2033,100
medium,253214,3.93,0,100
low,162830,2.18,0,100


In [0]:
print("=" * 60)
print("📊 MEDALLION ARCHITECTURE — DATA PIPELINE SUMMARY")
print("=" * 60)

for layer, table in [("Bronze", TABLES["raw_security_events"]), 
                      ("Silver", TABLES["normalized_events"]),
                      ("Gold", TABLES["enriched_alerts"])]:
    count = spark.table(table).count()
    print(f"  {layer:8s} | {table:55s} | {count:>10,} rows")

print("=" * 60)
print("✅ Lakeflow ingestion pipeline complete!")
print("\nNext step: Run 03_governance to add comments and tags")

📊 MEDALLION ARCHITECTURE — DATA PIPELINE SUMMARY
  Bronze   | rajarora_febar.soc_alert_triage.raw_security_events     |    500,000 rows
  Silver   | rajarora_febar.soc_alert_triage.silver_normalized_events |    500,000 rows
  Gold     | rajarora_febar.soc_alert_triage.gold_enriched_alerts    |    500,000 rows
✅ Lakeflow ingestion pipeline complete!

Next step: Run 03_governance to add comments and tags
